In [1]:
# ============================================================
# BranchyNet — ResNet-50 on CIFAR-10
# FINE-TUNING from baseline: __2__baseline_resnet50_cifar10.pth
# ============================================================

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import time, os, json, tempfile
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score)
import random

# ── REPRODUCIBILITY ───────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

g = torch.Generator()
g.manual_seed(SEED)

# ── CONFIG ────────────────────────────────────────────────────
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE  = 128
EPOCHS      = 20          # fine-tuning — backbone already converged
LR          = 1e-3        # lower LR — backbone needs gentle updates
NUM_CLASSES = 10
BASELINE_PATH = "../__2__baseline_resnet50_cifar10.pth"
SAVE_PATH     = "__3__branchynet_resnet50_cifar10.pth"

DATA_ROOT = "../data"

EXIT_THRESHOLDS = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
BRANCH_WEIGHTS  = [0.2, 0.3, 0.5]

CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"Using device: {DEVICE}")

# ── UTILITY FUNCTIONS ─────────────────────────────────────────
def disk_size_mb(model):
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        size = os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)
    return round(size, 4)

def measure_inference(model, device, batch_size=128,
                      runs_single=500, runs_batch=100):
    model.eval()
    use_cuda = device.type == "cuda"
    dummy_single = torch.randn(1, 3, 32, 32, device=device)
    dummy_batch  = torch.randn(batch_size, 3, 32, 32, device=device)

    # Single image — sync + perf_counter
    with torch.no_grad():
        for _ in range(50):
            model(dummy_single)
    if use_cuda:
        torch.cuda.synchronize()
    times = []
    with torch.no_grad():
        for _ in range(runs_single):
            if use_cuda:
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(dummy_single)
            if use_cuda:
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    inf_ms_single = np.mean(times) * 1000

    # Batch — CUDA events
    with torch.no_grad():
        for _ in range(10):
            model(dummy_batch)
    if use_cuda:
        torch.cuda.synchronize()
    if use_cuda:
        start_ev = torch.cuda.Event(enable_timing=True)
        end_ev   = torch.cuda.Event(enable_timing=True)
        batch_times = []
        with torch.no_grad():
            for _ in range(runs_batch):
                start_ev.record()
                model(dummy_batch)
                end_ev.record()
                torch.cuda.synchronize()
                batch_times.append(start_ev.elapsed_time(end_ev))
        inf_ms_batch = np.mean(batch_times)
    else:
        batch_times = []
        with torch.no_grad():
            for _ in range(runs_batch):
                t0 = time.perf_counter()
                model(dummy_batch)
                batch_times.append((time.perf_counter() - t0) * 1000)
        inf_ms_batch = np.mean(batch_times)

    return {
        "single_img_ms"      : round(inf_ms_single, 4),
        "batch128_ms"        : round(inf_ms_batch, 4),
        "per_img_ms"         : round(inf_ms_batch / batch_size, 4),
        "throughput_imgs_sec": round(batch_size / (inf_ms_batch / 1000), 1),
        "timing_method"      : "sync+perf_counter (single), CUDA events (batch)",
    }

def measure_adaptive_inference(model, device, threshold=0.8, runs=500):
    model.eval()
    use_cuda = device.type == "cuda"
    dummy = torch.randn(1, 3, 32, 32, device=device)
    with torch.no_grad():
        for _ in range(50):
            model.adaptive_forward(dummy, threshold=threshold)
    if use_cuda:
        torch.cuda.synchronize()
    times = []
    with torch.no_grad():
        for _ in range(runs):
            if use_cuda:
                torch.cuda.synchronize()
            t0 = time.perf_counter()
            model.adaptive_forward(dummy, threshold=threshold)
            if use_cuda:
                torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    return round(np.mean(times) * 1000, 4)

def compute_flops(model, device, input_size=(1, 3, 32, 32)):
    m = model.eval().to(device)
    total_flops = [0]
    hooks = []
    def conv_hook(module, inp, out):
        N, C_out, H_out, W_out = out.shape
        C_in = module.in_channels
        kH, kW = module.kernel_size if isinstance(module.kernel_size, tuple) \
                  else (module.kernel_size, module.kernel_size)
        total_flops[0] += 2 * N * C_out * H_out * W_out * (C_in // module.groups) * kH * kW
    def linear_hook(module, inp, out):
        total_flops[0] += 2 * inp[0].shape[0] * module.in_features * module.out_features
    for mod in m.modules():
        if isinstance(mod, nn.Conv2d):
            hooks.append(mod.register_forward_hook(conv_hook))
        elif isinstance(mod, nn.Linear):
            hooks.append(mod.register_forward_hook(linear_hook))
    with torch.no_grad():
        m(torch.randn(*input_size, device=device))
    for h in hooks:
        h.remove()
    return round(total_flops[0] / 1e9, 6)

# ── AUXILIARY BRANCH ──────────────────────────────────────────
class EarlyExitBranch(nn.Module):
    def __init__(self, input_channels, num_classes=10):
        super().__init__()
        self.branch = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.BatchNorm1d(input_channels),
            nn.Linear(input_channels, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.branch(x)

# ── BRANCHYNET MODEL ──────────────────────────────────────────
class BranchyResNet50(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Build architecture — no pretrained weights, we load manually
        backbone = models.resnet50(weights=None)
        backbone.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        backbone.maxpool = nn.Identity()
        backbone.fc      = nn.Linear(backbone.fc.in_features, num_classes)

        self.num_classes = num_classes
        self.stem    = nn.Sequential(backbone.conv1, backbone.bn1,
                                     backbone.relu, backbone.maxpool)
        self.layer1  = backbone.layer1
        self.layer2  = backbone.layer2
        self.layer3  = backbone.layer3
        self.layer4  = backbone.layer4
        self.avgpool = backbone.avgpool
        self.fc      = backbone.fc
        self.branch1 = EarlyExitBranch(256, num_classes)
        self.branch2 = EarlyExitBranch(512, num_classes)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x);  logits1 = self.branch1(x)
        x = self.layer2(x);  logits2 = self.branch2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits3 = self.fc(x)
        return logits1, logits2, logits3

    @torch.no_grad()
    def adaptive_forward(self, x, threshold=0.8):
        B      = x.size(0)
        device = x.device
        final_logits     = torch.zeros(B, self.num_classes, device=device)
        exit_idx         = torch.full((B,), 2, dtype=torch.long, device=device)
        remaining_global = torch.arange(B, device=device)

        x = self.stem(x)
        x = self.layer1(x)
        logits1 = self.branch1(x)
        conf1   = torch.softmax(logits1, dim=1).max(dim=1).values
        done1   = conf1 >= threshold
        if done1.any():
            g_idx = remaining_global[done1]
            final_logits[g_idx] = logits1[done1]
            exit_idx[g_idx]     = 0
        keep = ~done1
        if not keep.any():
            return final_logits, exit_idx
        x                = x[keep]
        remaining_global = remaining_global[keep]

        x = self.layer2(x)
        logits2 = self.branch2(x)
        conf2   = torch.softmax(logits2, dim=1).max(dim=1).values
        done2   = conf2 >= threshold
        if done2.any():
            g_idx = remaining_global[done2]
            final_logits[g_idx] = logits2[done2]
            exit_idx[g_idx]     = 1
        keep = ~done2
        if not keep.any():
            return final_logits, exit_idx
        x                = x[keep]
        remaining_global = remaining_global[keep]

        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        logits3 = self.fc(x)
        final_logits[remaining_global] = logits3
        return final_logits, exit_idx

# ── BUILD MODEL AND LOAD BASELINE WEIGHTS ─────────────────────
model = BranchyResNet50(num_classes=NUM_CLASSES).to(DEVICE)

baseline_state = torch.load(BASELINE_PATH, map_location=DEVICE)
own_state = model.state_dict()
loaded, skipped = 0, 0
for k, v in baseline_state.items():
    # Remap stem keys
    if   k == "conv1.weight":            rk = "stem.0.weight"
    elif k == "bn1.weight":              rk = "stem.1.weight"
    elif k == "bn1.bias":                rk = "stem.1.bias"
    elif k == "bn1.running_mean":        rk = "stem.1.running_mean"
    elif k == "bn1.running_var":         rk = "stem.1.running_var"
    elif k == "bn1.num_batches_tracked": rk = "stem.1.num_batches_tracked"
    else:                                rk = k
    if rk in own_state:
        own_state[rk].copy_(v)
        loaded += 1
    else:
        skipped += 1

model.load_state_dict(own_state)
print(f"✓ Baseline backbone loaded — {loaded} tensors copied, {skipped} skipped")
print(f"  branch1 and branch2 are randomly initialized")

total_params = sum(p.numel() for p in model.parameters())
print(f"  Total parameters: {total_params:,}")

# ── DATA ──────────────────────────────────────────────────────
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR_MEAN, CIFAR_STD)
])

train_set = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=True,
                                          download=True, transform=transform_train)
test_set  = torchvision.datasets.CIFAR10(root=DATA_ROOT, train=False,
                                          download=True, transform=transform_test)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE,
                                            shuffle=True, num_workers=0,
                                            worker_init_fn=seed_worker, generator=g)
test_loader  = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE,
                                            shuffle=False, num_workers=0)
print(f"Train: {len(train_set)} | Test: {len(test_set)}")

# ── LOSS & OPTIMIZER ──────────────────────────────────────────
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.SGD(model.parameters(), lr=LR,
                             momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

def branchynet_loss(logits_list, labels, weights=BRANCH_WEIGHTS):
    return sum(w * criterion(l, labels) for w, l in zip(weights, logits_list))

def train_epoch(model, loader, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        logits1, logits2, logits3 = model(inputs)
        loss = branchynet_loss([logits1, logits2, logits3], labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += logits3.argmax(1).eq(labels).sum().item()
        total   += labels.size(0)
    return total_loss / len(loader), correct / total

def evaluate_standard(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            _, _, logits3 = model(inputs)
            correct += logits3.argmax(1).eq(labels).sum().item()
            total   += labels.size(0)
    return correct / total

# ── TRAINING LOOP ─────────────────────────────────────────────
best_val_acc = 0.0
train_accs, val_accs, train_losses = [], [], []

print("\n" + "="*60)
print("FINE-TUNING — BranchyNet on CIFAR-10 (from baseline)")
print("="*60)

for epoch in range(EPOCHS):
    loss, train_acc = train_epoch(model, train_loader, optimizer)
    val_acc         = evaluate_standard(model, test_loader)
    scheduler.step()

    train_accs.append(train_acc)
    val_accs.append(val_acc)
    train_losses.append(loss)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), SAVE_PATH)
        marker = " ← best saved"
    else:
        marker = ""

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | Loss: {loss:.4f} | "
          f"Train: {train_acc:.4f} | Val: {val_acc:.4f}{marker}")

print(f"\nBest validation accuracy (final exit): {best_val_acc:.4f}")

# ── FULL EVALUATION ───────────────────────────────────────────
print("\n" + "="*60)
print("FULL EVALUATION — final exit only")
print("="*60)

model.load_state_dict(torch.load(SAVE_PATH, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(DEVICE)
        _, _, logits3 = model(inputs)
        all_preds.extend(logits3.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc_final  = accuracy_score(all_labels, all_preds)
prec_final = precision_score(all_labels, all_preds, average='macro', zero_division=0)
rec_final  = recall_score(all_labels, all_preds, average='macro', zero_division=0)
f1_final   = f1_score(all_labels, all_preds, average='macro', zero_division=0)

print(f"  Accuracy          : {acc_final:.4f}")
print(f"  Precision (macro) : {prec_final:.4f}")
print(f"  Recall    (macro) : {rec_final:.4f}")
print(f"  F1-score  (macro) : {f1_final:.4f}")

# ── ADAPTIVE THRESHOLD SWEEP ──────────────────────────────────
print("\n" + "="*60)
print("ADAPTIVE COMPUTATION — Threshold Sweep")
print("="*60)

adaptive_results = []
for tau in EXIT_THRESHOLDS:
    preds_list, labels_list, exit_idx_list = [], [], []
    t_start = time.time()
    model.eval()
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(DEVICE)
            logits, exit_idx = model.adaptive_forward(inputs, threshold=tau)
            preds_list.extend(logits.argmax(1).cpu().numpy())
            labels_list.extend(labels.numpy())
            exit_idx_list.extend(exit_idx.cpu().numpy())
    t_end = time.time()

    preds_arr    = np.array(preds_list)
    labels_arr   = np.array(labels_list)
    exit_idx_arr = np.array(exit_idx_list)
    n            = len(labels_arr)

    acc  = accuracy_score(labels_arr, preds_arr)
    prec = precision_score(labels_arr, preds_arr, average='macro', zero_division=0)
    rec  = recall_score(labels_arr, preds_arr, average='macro', zero_division=0)
    f1   = f1_score(labels_arr, preds_arr, average='macro', zero_division=0)

    frac_exit1  = (exit_idx_arr == 0).mean()
    frac_exit2  = (exit_idx_arr == 1).mean()
    frac_exit3  = (exit_idx_arr == 2).mean()
    avg_time_ms = (t_end - t_start) / n * 1000

    adaptive_results.append({
        "threshold"  : tau,
        "accuracy"   : round(acc,  6),
        "precision"  : round(prec, 6),
        "recall"     : round(rec,  6),
        "f1"         : round(f1,   6),
        "frac_exit1" : round(frac_exit1, 4),
        "frac_exit2" : round(frac_exit2, 4),
        "frac_exit3" : round(frac_exit3, 4),
        "avg_time_ms": round(avg_time_ms, 4),
    })
    print(f"  τ={tau:.2f} | Acc={acc:.4f} | F1={f1:.4f} | "
          f"Exit1={frac_exit1:.1%} Exit2={frac_exit2:.1%} Exit3={frac_exit3:.1%} | "
          f"Time={avg_time_ms:.4f}ms/sample")

best_adaptive = max(adaptive_results, key=lambda r: r["accuracy"])

# ── MODEL COMPLEXITY ──────────────────────────────────────────
print("\n" + "="*60)
print("MODEL COMPLEXITY METRICS")
print("="*60)

flops_g  = compute_flops(model, DEVICE)
infer    = measure_inference(model, DEVICE)
size_mb  = disk_size_mb(model)
inf_ms_adapt = measure_adaptive_inference(model, DEVICE, threshold=0.8)

print(f"  Parameters               : {total_params:,}")
print(f"  Model size               : {size_mb:.4f} MB")
print(f"  FLOPs (full)             : {flops_g:.4f} G")
print(f"  Inference (single, full) : {infer['single_img_ms']:.3f} ms")
print(f"  Inference (batch128)     : {infer['batch128_ms']:.2f} ms "
      f"({infer['throughput_imgs_sec']:.1f} img/s)")
print(f"  Inference (τ=0.8 adapt)  : {inf_ms_adapt:.3f} ms")

# ── SAVE METRICS JSON ─────────────────────────────────────────
branchynet_metrics = {
    "accuracy"                    : round(acc_final,  6),
    "precision"                   : round(prec_final, 6),
    "recall"                      : round(rec_final,  6),
    "f1"                          : round(f1_final,   6),
    "params"                      : total_params,
    "size_mb"                     : size_mb,
    "flops_G"                     : flops_g,
    "inference_ms"                : infer,
    "method"                      : "early_exit",
    "variant"                     : "BranchyNet_ResNet50",
    "dataset"                     : "CIFAR-10",
    "training_mode"               : "fine_tuning_from_baseline",
    "baseline_path"               : BASELINE_PATH,
    "epochs"                      : EPOCHS,
    "lr"                          : LR,
    "branch_weights"              : BRANCH_WEIGHTS,
    "num_exits"                   : 3,
    "exit_positions"              : ["after layer1 (256ch)",
                                     "after layer2 (512ch)",
                                     "final fc (2048ch)"],
    "exit_thresholds_tested"      : EXIT_THRESHOLDS,
    "inference_ms_adaptive_tau08" : inf_ms_adapt,
    "adaptive_threshold_results"  : adaptive_results,
    "best_adaptive_result"        : best_adaptive,
    "saved_model_path"            : os.path.abspath(SAVE_PATH),
}

with open("__3__branchynet_cifar10_metrics.json", "w") as f:
    json.dump(branchynet_metrics, f, indent=2)
print("\n✓ Metrics saved → __3__branchynet_cifar10_metrics.json")

# ── COMPARISON SUMMARY ────────────────────────────────────────
print("\n" + "="*60)
print("COMPARISON: Baseline vs BranchyNet")
print("="*60)

baseline_json = "__2__baseline_metrics.json"
if os.path.exists(baseline_json):
    with open(baseline_json) as f:
        base = json.load(f)
    scalar_keys = ["accuracy", "precision", "recall", "f1", "params", "size_mb", "flops_G"]
    print(f"\n  {'Metric':<22} {'Baseline':>12} {'BranchyNet':>12} {'Δ':>10}")
    print("  " + "-"*58)
    for k in scalar_keys:
        bv = base.get(k, float('nan'))
        mv = branchynet_metrics.get(k, float('nan'))
        if not isinstance(bv, (int, float)) or not isinstance(mv, (int, float)):
            continue
        delta = mv - bv
        sign  = "+" if delta > 0 else ""
        if k == "params":
            print(f"  {k:<22} {bv:>12,} {mv:>12,} {sign}{delta:>9,.0f}")
        else:
            print(f"  {k:<22} {bv:>12.4f} {mv:>12.4f} {sign}{delta:>9.4f}")
    full_ms = infer['single_img_ms']
    print(f"\n  Inference (single, full) : {full_ms:.3f} ms")
    print(f"  Inference (τ=0.8 adapt)  : {inf_ms_adapt:.3f} ms "
          f"({(1 - inf_ms_adapt/full_ms)*100:+.1f}% vs full)")
    print(f"\n  Best adaptive (τ={best_adaptive['threshold']}):")
    print(f"    Accuracy : {best_adaptive['accuracy']:.4f}")
    print(f"    Exit1    : {best_adaptive['frac_exit1']:.1%}")
    print(f"    Exit2    : {best_adaptive['frac_exit2']:.1%}")
    print(f"    Exit3    : {best_adaptive['frac_exit3']:.1%}")
else:
    print(f"  (baseline JSON not found at {baseline_json})")

# ── CHECKPOINT ────────────────────────────────────────────────
torch.save({
    "model_state_dict": model.state_dict(),
    "config": {
        "model_name"     : "BranchyResNet50_CIFAR10_finetuned",
        "num_classes"    : NUM_CLASSES,
        "input_size"     : [3, 32, 32],
        "num_exits"      : 3,
        "branch_weights" : BRANCH_WEIGHTS,
        "training_mode"  : "fine_tuning_from_baseline",
        "normalization"  : {"mean": CIFAR_MEAN, "std": CIFAR_STD},
    },
    "classes": CIFAR10_CLASSES,
}, "__3__branchynet_cifar10_checkpoint.pth")

print("\n" + "="*60)
print("BRANCHYNET CIFAR-10 FINE-TUNING — COMPLETE")
print("="*60)
print(f"  Weights    → {SAVE_PATH}")
print(f"  Checkpoint → __3__branchynet_cifar10_checkpoint.pth")
print(f"  Metrics    → __3__branchynet_cifar10_metrics.json")

Using device: cuda
✓ Baseline backbone loaded — 320 tensors copied, 0 skipped
  branch1 and branch2 are randomly initialized
  Total parameters: 23,623,518


e:\baseline_resnet50_cifar10\env\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train: 50000 | Test: 10000

FINE-TUNING — BranchyNet on CIFAR-10 (from baseline)
Epoch  1/20 | Loss: 1.2720 | Train: 0.9873 | Val: 0.9312 ← best saved
Epoch  2/20 | Loss: 1.1395 | Train: 0.9867 | Val: 0.9316 ← best saved
Epoch  3/20 | Loss: 1.0817 | Train: 0.9866 | Val: 0.9306
Epoch  4/20 | Loss: 1.0463 | Train: 0.9853 | Val: 0.9306
Epoch  5/20 | Loss: 1.0189 | Train: 0.9859 | Val: 0.9300
Epoch  6/20 | Loss: 0.9982 | Train: 0.9859 | Val: 0.9285
Epoch  7/20 | Loss: 0.9805 | Train: 0.9861 | Val: 0.9291
Epoch  8/20 | Loss: 0.9670 | Train: 0.9859 | Val: 0.9293
Epoch  9/20 | Loss: 0.9536 | Train: 0.9872 | Val: 0.9300
Epoch 10/20 | Loss: 0.9443 | Train: 0.9866 | Val: 0.9303
Epoch 11/20 | Loss: 0.9345 | Train: 0.9867 | Val: 0.9299
Epoch 12/20 | Loss: 0.9266 | Train: 0.9875 | Val: 0.9294
Epoch 13/20 | Loss: 0.9206 | Train: 0.9873 | Val: 0.9302
Epoch 14/20 | Loss: 0.9143 | Train: 0.9877 | Val: 0.9300
Epoch 15/20 | Loss: 0.9100 | Train: 0.9885 | Val: 0.9300
Epoch 16/20 | Loss: 0.9067 | Train: 0.